In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import gzip
import warnings
warnings.filterwarnings('ignore')

columns = [
    'karl_id', 'host_name', 'model_name', 'hardware_make', 'karl_last_seen',
    'auth_username', 'serial_number', 'group_id', 'tenant_id', 'platform',
    'metric_category', 'measure_name', 'time', 'p90_processor_time',
    'avg_processor_time', 'max_cpu_usage', 'p90_memory_utilization',
    'avg_memory_utilization', 'max_memory_usage', 'p10_battery_health',
    'avg_battery_health', 'cpu_count', 'memory_count', 'memory_size_gb',
    'driver_vendor', 'os', 'wifi_mac_add', 'driver_version', 'driver_date',
    'os_version', 'driver', 'agent_id', 'performance_status', 'device_status',
    'max_battery_temperature', 'avg_battery_temperature', 'p90_battery_temperature',
    'avg_cpu_temp', 'p90_cpu_temp', 'avg_battery_discharge', 'p90_battery_discharge',
    'avg_boot_time', 'p90_boot_time', 'uptime_days', 'total_app_crash'
]

# Load sample
chunk_size = 100000
sample_data = []
with gzip.open('000.gz', 'rt') as f:
    for i, chunk in enumerate(pd.read_csv(f, sep='|', names=columns, chunksize=chunk_size)):
        sample_data.append(chunk)
        if i >= 4:
            break

df = pd.concat(sample_data, ignore_index=True)

numeric_cols = [
    'avg_processor_time', 'max_cpu_usage', 'avg_memory_utilization',
    'max_memory_usage', 'avg_battery_health', 'cpu_count', 'memory_size_gb',
    'avg_cpu_temp', 'avg_boot_time', 'p90_boot_time',
    'uptime_days', 'total_app_crash', 'p90_cpu_temp'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
#print(f"Loaded {len(df)} rows")

# narrow down
feature_cols = ['avg_processor_time','max_cpu_usage', 'avg_memory_utilization',
'avg_battery_health',
'uptime_days', 'p90_cpu_temp']

model_df = df[feature_cols + ['total_app_crash']].dropna()

# 90th percentile
threshold = model_df['total_app_crash'].quantile(0.90)

model_df = model_df[model_df['total_app_crash'] >= threshold]

X = model_df[feature_cols]
y = model_df['total_app_crash']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Training model
#print("\nTraining Random Forest Regressor...")

rf_model = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

#print("\n=== REGRESSION PERFORMANCE ===")
#print(f"MSE: {mse:.2f}")
#print(f"MAE: {mae:.2f}")
#print(f"R²: {r2:.3f}")

# fake feature variables to run through the model - changing one features values to see where there's predicted spikes
def get_feature_response(model, X, feature, grid_size=50):
    X_temp = X.copy()
    
    # baseline = median of all features
    baseline = X.median()
    
    values = np.linspace(X[feature].min(), X[feature].max(), grid_size)
    preds = []

    for val in values:
        row = baseline.copy()
        row[feature] = val
        preds.append(model.predict([row])[0])

    return values, np.array(preds)

def plot_with_thresholds(model, X, feature):
    values, preds = get_feature_response(model, X, feature)

    # compute gradient 
    gradient = np.gradient(preds)

    # threshold = top spikes 
    spike_idx = np.where(gradient > np.percentile(gradient, 90))[0]

    #plt.figure(figsize=(10,6))
    #plt.plot(values, preds, label='Predicted crashes')
    
    # mark spike points
    #plt.scatter(values[spike_idx], preds[spike_idx], color='red', label='Spike points')

    #for i in spike_idx:
        #plt.axvline(values[i], color='red', linestyle='--', alpha=0.3)

    # Pulling the thresholds
    if (feature == 'avg_processor_time'):
        idx = spike_idx[4]
        val = values[idx]
        spike_dict[feature] = val
    if (feature == 'max_cpu_usage'):
        idx = spike_idx[4]
        val = values[idx]
        spike_dict[feature] = val
    if (feature == 'avg_battery_health'):
        idx = spike_idx[2]
        val = values[idx]
        spike_dict[feature] = val
    if (feature == 'avg_memory_utilization'):
        idx = spike_idx[4]
        val = values[idx]
        spike_dict[feature] = val
    if (feature == 'uptime_days'):
        idx = spike_idx[3]
        val = values[idx]
        spike_dict[feature] = val
    if (feature == 'p90_cpu_temp'):
        idx = spike_idx[4]
        val = values[idx]
        spike_dict[feature] = val

    #plt.title(f'Crash Prediction vs {feature}')
    #plt.xlabel(feature)
    #plt.ylabel('Predicted total_app_crash')
    #plt.legend()
    #plt.grid()
    #plt.show()

    #print(f"\nPotential thresholds for {feature}:")
    #print(values[spike_idx])
    #print(preds[spike_idx])
    #return {feature, sp}

spike_dict = {}
for feature in feature_cols:
    plot_with_thresholds(rf_model, X_train, feature)

# Spike dict is the {feature : threshold}
print(spike_dict)
